# LoRA layer-wise factual predictions — Kaggle launcher

Thin launcher: all logic lives in the repo (editable via IDE, reviewable in PRs);
this notebook only clones and runs it.

Setup: **Settings → Accelerator → GPU (T4 x2 or P100)**, Internet **On**.
Use *Save & Run All (Commit)* for free background execution.
Outputs land in `/kaggle/working/outputs`; download `outputs/results/` when done.

In [ ]:
# REF can be a branch for iteration, or a commit SHA to pin a paper-grade run.
REF = "main"   # switch to "main" after the PR merges
CONFIG = "configs/default.yaml"   # configs/dev.yaml for a fast smoke test
STAGES = "all"                    # or e.g. "analyze,patch" to resume

import os
if os.path.exists("nlp-project"):
    !cd nlp-project && git fetch && git checkout {REF} && git pull --ff-only || true
else:
    !git clone https://github.com/hadasy-tau/nlp-project.git && cd nlp-project && git checkout {REF}

In [ ]:
!pip install -q -r nlp-project/requirements.txt
!pip install -q -e nlp-project
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE — enable an accelerator!")

In [ ]:
# Model and data are configurable — add --set overrides as needed, e.g.:
#   --set model.name=EleutherAI/pythia-1b-deduped
#   --set data.dataset=some/other-dataset --set data.fields.relation=predicate
!cd nlp-project && python -m lora_lens.run --config {CONFIG} --stages {STAGES} \
    --set output_dir=/kaggle/working/outputs

In [ ]:
# Bundle results for download from the Output tab
!cd /kaggle/working && zip -qr results.zip outputs/results outputs/config_resolved.yaml outputs/lora/training_log.csv
print("Download /kaggle/working/results.zip from the Output tab.")